# SpMV kernel selector: Colab runner

Thin runner for the Member 3 selector and evaluation pipeline. **All logic lives in the repository**
(`selector/`, `evaluation/`, `scripts/`); this notebook only calls it.

- **Runtime: CPU is enough** (Runtime > Change runtime type > None / CPU). Nothing here uses a GPU.
- Run the cells **top to bottom**. To start over, run from the top again (it re-creates `/content/spmv`).
- With `USE_SYNTHETIC_DEMO = True` the data is **fabricated demo data**. Every output is stamped
  `SYNTHETIC DATA - NOT A RESULT`. Set it to `False` only when real `kernel_timings.csv` exists in the project.
- Full protocol: `docs/selector_evaluation.md`.

In [ ]:
# ---- 1. Settings (the only cell you may need to edit) ----
SOURCE = "git"          # "git": clone from GitHub (branch must be pushed)  |  "drive": copy a project folder from Google Drive
REPO_URL = "https://github.com/MayeraaSingh/Feature-Based-GPU-SpMV-Kernel-Selection.git"
BRANCH = "selector-evaluation"
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/Feature-Based-GPU-SpMV-Kernel-Selection"   # used when SOURCE = "drive"
USE_SYNTHETIC_DEMO = True   # True: demo data (NOT a result)  |  False: real CSVs already inside the project

In [ ]:
# ---- 2. Get the code into /content/spmv ----
import os, shutil
os.chdir("/content")
shutil.rmtree("/content/spmv", ignore_errors=True)
if SOURCE == "git":
    !git clone --branch {BRANCH} {REPO_URL} /content/spmv
else:
    from google.colab import drive
    drive.mount("/content/drive")
    shutil.copytree(DRIVE_PROJECT_DIR, "/content/spmv",
                    ignore=shutil.ignore_patterns(".venv", "venv", "results", ".git", "__pycache__"))
%cd /content/spmv

In [ ]:
# ---- 3. Install dependencies (Colab already has most of them) ----
!pip install -q -r requirements.txt
!python --version

In [ ]:
# ---- 4. Run the test-suite (should end with "passed", no failures) ----
!python -m pytest tests/ -q

In [ ]:
# ---- 5. Choose the data ----
import time
RUN_ID = "colab-" + time.strftime("%Y%m%d-%H%M%S")
if USE_SYNTHETIC_DEMO:
    !python scripts/make_synthetic_demo.py
    CONFIG = "data/synthetic_demo/demo_config.yaml"
    EXTRA = "--allow-synthetic"
else:
    CONFIG = "configs/selector_config.yaml"
    EXTRA = ""
print("run id:", RUN_ID, "| config:", CONFIG)

In [ ]:
# ---- 6. Train on the train split, tune on the validation split ----
!python scripts/train_selector.py --config {CONFIG} {EXTRA} --run-id {RUN_ID}

In [ ]:
# ---- 7. Evaluate the frozen model on the held-out test split ----
!python scripts/evaluate_generalization.py --config {CONFIG} {EXTRA} --run-dir results/{RUN_ID}

In [ ]:
# ---- 8. Show the results ----
import pandas as pd
from IPython.display import display, Image, Markdown
run = f"results/{RUN_ID}"
display(Markdown(open(f"{run}/summary_overall.md").read()))
display(pd.read_csv(f"{run}/per_matrix_test.csv"))
for name in sorted(os.listdir(f"{run}/figures")):
    display(Image(f"{run}/figures/{name}"))

In [ ]:
# ---- 9. Download the results (the model file is left out on purpose) ----
!zip -qr /content/{RUN_ID}.zip {run} -x "*.joblib"
from google.colab import files
files.download(f"/content/{RUN_ID}.zip")